# Working with Databases in Python: An Exam Walkthrough

This notebook walks you through the kinds of tasks that appear in the Database Methods practical exam, using a fictional record shop called **Moonrise Records** as our working example. By the end, you will have practiced every type of question the exam contains, with explanations of what each line of code is actually doing.

---

## How the Libraries Fit Together

Before writing a single line of code, it is worth understanding how the tools we use relate to one another. Think of it like a kitchen:

- **SQLite** is the filing cabinet where data lives. It stores everything in a single file on disk (ending in `.db`). The file can hold multiple tables, just like a filing cabinet holds multiple folders.
- **sqlite3** is the Python library that lets you open that filing cabinet, read from it, and write to it. You speak to it in SQL.
- **pandas** is a library for working with data as tables (called DataFrames) inside Python. It has its own tools for reading from and writing to SQLite, which means you can pull a whole table into Python with one line, work on it there, and push it back when you are done.
- **ipywidgets** builds interactive controls — text boxes, dropdowns, buttons — inside a Jupyter notebook. We use these to make data entry forms that insert records into the database when a button is clicked.
- **matplotlib** draws charts from data stored in Python variables or DataFrames.

These libraries do not compete with one another — they each have a job, and they pass data back and forth. A typical exam task might go: *connect to database → run SQL query → load result into pandas → pass to matplotlib to draw a chart*.

---

## What is a .db file?

A `.db` file is simply a SQLite database stored as a single file on your computer. It is not a spreadsheet and it is not a CSV — it is a proper relational database that can hold multiple tables and complex relationships between them. You open it from Python using `sqlite3.connect()`. If the file does not exist yet, SQLite creates it for you automatically.

---

## Setting Up: Imports and Database Creation

The cell below imports every library you will need throughout this notebook. In the exam, these imports will be provided for you in the first cell — your job is just to run that cell before doing anything else.

In [ ]:
# sqlite3 is a standard Python library — no installation needed
# It lets Python talk to SQLite databases
import sqlite3

# pandas is the data table library
# 'pd' is the conventional short name everyone uses
import pandas as pd

# matplotlib is the charting library
# pyplot is the part we actually use for drawing charts
import matplotlib.pyplot as plt

# ipywidgets gives us interactive controls (text boxes, buttons, etc.)
import ipywidgets as widgets
from ipywidgets import VBox, HBox

# This makes charts appear inside the notebook rather than in a separate window
%matplotlib inline

print("All libraries loaded successfully.")

### Creating the Practice Database

The cell below creates the Moonrise Records database and populates it with sample data. In the real exam, this file is already provided — you do not need to create it. Here we build it ourselves so that the rest of the notebook works.

Read through it anyway: you will see the CREATE TABLE and INSERT INTO syntax that appears in the exam.

In [ ]:
# sqlite3.connect() opens the database file.
# If 'moonrise_records.db' doesn't exist on disk, SQLite creates it here.
conn = sqlite3.connect('moonrise_records.db')

# A cursor is an object that lets us send SQL commands to the database.
# Think of it as the pen that writes SQL into the filing cabinet.
cursor = conn.cursor()

# DROP TABLE IF EXISTS means: delete this table if it already exists.
# This prevents errors if we run the cell more than once.
cursor.execute("DROP TABLE IF EXISTS albums")
cursor.execute("DROP TABLE IF EXISTS artists")

# CREATE TABLE defines the structure (columns and data types) of a table.
# The data types used here are TEXT (for words), INTEGER (whole numbers),
# and REAL (decimal numbers).
cursor.execute("""
    CREATE TABLE artists (
        artist_id   INTEGER PRIMARY KEY,
        name        TEXT,
        country     TEXT,
        genre       TEXT
    )
""")

cursor.execute("""
    CREATE TABLE albums (
        album_id    INTEGER PRIMARY KEY,
        title       TEXT,
        artist_id   INTEGER,
        year        INTEGER,
        price       REAL,
        stock       INTEGER
    )
""")

# INSERT INTO adds rows to a table.
# The VALUES list must match the column order in CREATE TABLE.
artists_data = [
    (1, 'Fleetwood Mac',  'USA',  'Rock'),
    (2, 'Siouxsie and the Banshees', 'UK', 'Post-Punk'),
    (3, 'Caetano Veloso', 'Brazil', 'MPB'),
    (4, 'Fela Kuti',      'Nigeria', 'Afrobeat'),
    (5, 'Clannad',        'Ireland', 'Folk'),
    (6, 'Björk',          'Iceland', 'Art Pop'),
    (7, 'Miles Davis',    'USA',  'Jazz'),
]

# executemany() inserts a list of rows in one go
cursor.executemany("INSERT INTO artists VALUES (?, ?, ?, ?)", artists_data)

albums_data = [
    (1,  'Rumours',                  1, 1977, 24.99,  8),
    (2,  'Tusk',                     1, 1979, 19.99,  3),
    (3,  'The Scream',               2, 1978, 22.50,  5),
    (4,  'Juju',                     2, 1981, 21.00,  2),
    (5,  'Transa',                   3, 1997, 18.50,  6),
    (6,  'Zombie',                   4, 1977, 26.00, 10),
    (7,  'Expensive Shit',           4, 1975, 23.50,  4),
    (8,  'Magical Ring',             5, 1983, 17.99,  7),
    (9,  'Homogenic',                6, 1997, 27.50,  9),
    (10, 'Kind of Blue',             7, 1959, 29.99, 12),
    (11, 'Bitches Brew',             7, 1970, 32.00,  5),
]

cursor.executemany("INSERT INTO albums VALUES (?, ?, ?, ?, ?, ?)", albums_data)

# conn.commit() saves all changes to the .db file.
# Without this line, inserts and updates are temporary and will be lost.
conn.commit()

print("Database created: moonrise_records.db")
print("Tables: artists, albums")

---

## Part 1: Exploring the Database (Task 1 in the Exam)

The first task in the exam always asks you to explore a database you have been given. You will typically need to:

- Inspect the structure of a table (its columns and data types)
- Count how many records it contains
- Display a preview of its contents

### 1a: Viewing Column Structure with PRAGMA

`PRAGMA table_info(table_name)` is a special SQLite command that lists every column in a table along with its data type and whether it allows empty values. This is how you look at the *structure* of a table rather than its *contents*.

In [ ]:
# pd.read_sql() runs a SQL query and returns the result as a pandas DataFrame.
# It takes two arguments: the SQL query (as a string) and the connection to use.

# PRAGMA table_info tells us the column names and types in the albums table.
structure = pd.read_sql("PRAGMA table_info(albums)", conn)

# display() shows a formatted table in Jupyter.
# It works with DataFrames, plain text, and other objects.
display(structure)

The output shows one row per column. The columns you care most about are:
- **name** — the column name
- **type** — the data type (TEXT, INTEGER, REAL, etc.)
- **notnull** — 1 means the field cannot be left empty; 0 means it can
- **pk** — 1 means this is the primary key

### 1b: Counting Records with SELECT COUNT

`COUNT(*)` is a SQL aggregate function that counts how many rows are in a table. The `AS` keyword gives the result column a friendlier name.

In [ ]:
# This query counts every row in the albums table.
# COUNT(*) means "count all rows, regardless of any conditions".
count_result = pd.read_sql("SELECT COUNT(*) AS total_albums FROM albums", conn)

display(count_result)

### 1c: Previewing Table Contents

`SELECT * FROM table_name` retrieves every row and every column. Once you have the result in a DataFrame, `.head(5)` shows only the first five rows — useful when a table has hundreds or thousands of records.

In [ ]:
# SELECT * means "select all columns".
# The result comes back as a pandas DataFrame called df_albums.
# The 'df_' prefix is a convention: it reminds us this variable is a DataFrame.
df_albums = pd.read_sql("SELECT * FROM albums", conn)

# .head(5) shows the first 5 rows. Change the number to see more or fewer.
display(df_albums.head(5))

---

## Part 2: Writing SQL Queries (Task 2 in the Exam)

Task 2 asks you to retrieve specific subsets of data using SQL. The key tools here are:

- **WHERE** — filters rows based on a condition
- **ORDER BY** — sorts the results
- **Comparison operators** — `>`, `<`, `>=`, `<=`, `=`, `!=`
- **Logical operators** — `AND`, `OR`

All of these queries use the same pattern: write the SQL as a Python string, pass it to `pd.read_sql()`, and display the result.

### 2a: Filtering with WHERE

In [ ]:
# WHERE price > 25 keeps only rows where the price column is greater than 25.
# Note: the SQL is written as a Python string using triple quotes,
# which lets the query span multiple lines and stay readable.

expensive_albums = pd.read_sql("""
    SELECT title, price
    FROM albums
    WHERE price > 25
""", conn)

display(expensive_albums)

### 2b: Filtering with a Text Condition

When filtering on a TEXT column, the value you compare against must be written in single quotes inside the SQL string. Notice the difference between the outer Python quotes and the inner SQL quotes.

In [ ]:
# WHERE genre = 'Jazz' — the value 'Jazz' is in single quotes because it is a SQL string.
# The entire query is wrapped in Python triple quotes.

jazz_artists = pd.read_sql("""
    SELECT name, country
    FROM artists
    WHERE genre = 'Jazz'
""", conn)

display(jazz_artists)

### 2c: Sorting with ORDER BY

`ORDER BY column_name ASC` sorts results in ascending (A to Z, smallest to largest) order. `DESC` reverses it. If you leave out ASC/DESC, SQLite defaults to ascending.

In [ ]:
# ORDER BY year ASC sorts from oldest to newest.
# ORDER BY year DESC would sort from newest to oldest.

albums_by_year = pd.read_sql("""
    SELECT title, year, price
    FROM albums
    ORDER BY year ASC
""", conn)

display(albums_by_year)

### 2d: Combining Conditions with AND

You can chain multiple conditions with AND or OR. Both conditions must be true (AND) or at least one must be true (OR) for a row to appear in the results.

In [ ]:
# Both conditions must be true:
# the album must have been released after 1990 AND have more than 5 copies in stock.

recent_and_stocked = pd.read_sql("""
    SELECT title, year, stock
    FROM albums
    WHERE year > 1990
    AND stock > 5
""", conn)

display(recent_and_stocked)

### Practice Questions — Part 2

Try writing the following queries yourself. Each one uses the same pattern as the examples above.

In [ ]:
# Practice 2A:
# Find all albums where stock is less than 5.
# Show the title and stock columns only.
# Your code here:


In [ ]:
# Practice 2B:
# Show all artists from the UK.
# Sort the results alphabetically by name.
# Your code here:


In [ ]:
# Practice 2C:
# Find all albums released before 1980.
# Show the title, year, and price. Sort by price descending (most expensive first).
# Your code here:


---

## Part 3: Building a Data Entry Form with ipywidgets (Task 3 in the Exam)

This is the part that students find most challenging, but the structure is always the same. Read through this section carefully — once you understand the pattern, you can apply it to any form.

### What ipywidgets are doing

Each widget is an interactive element that appears in the notebook as a visual control. The user types into it or picks from a list, and the widget stores that value in its `.value` property. When the button is clicked, a function runs that reads `.value` from each widget and uses those values to build a SQL INSERT statement.

### The widgets we use

| Widget | What it creates | Example |
|---|---|---|
| `widgets.Text()` | A text input box | For names, titles |
| `widgets.IntText()` | A number input (whole numbers) | For years, counts |
| `widgets.FloatText()` | A number input (decimals) | For prices |
| `widgets.Dropdown()` | A dropdown list | For choosing from fixed options |
| `widgets.Button()` | A clickable button | For submitting the form |
| `widgets.Output()` | An area to display messages | For confirming the insert worked |

### Step-by-step: Building a form that adds a new artist

In [ ]:
# ─── Step 1: Create the input widgets ───────────────────────────────────────
# Each widget is created with a description= label that appears to the left of the field.
# style= controls how wide the label is — {'description_width': 'initial'} means
# 'as wide as needed', which stops long labels from being cut off.

name_input = widgets.Text(
    description='Artist name:',
    style={'description_width': 'initial'}
)

country_input = widgets.Text(
    description='Country:',
    style={'description_width': 'initial'}
)

genre_input = widgets.Dropdown(
    description='Genre:',
    options=['Rock', 'Jazz', 'Folk', 'Post-Punk', 'Afrobeat', 'Art Pop', 'MPB', 'Classical', 'Electronic'],
    style={'description_width': 'initial'}
)

# ─── Step 2: Create the button and output area ───────────────────────────────
# The button triggers an action when clicked.
# The output widget is where we display a confirmation message after the insert.
submit_button = widgets.Button(
    description='Add Artist',
    button_style='success'   # 'success' makes it green; other options: 'info', 'warning', 'danger'
)

# Output() creates an invisible box that will show text when we write to it.
output = widgets.Output()

# ─── Step 3: Define what happens when the button is clicked ─────────────────
# This function runs every time the button is clicked.
# The 'b' parameter is automatically passed in by ipywidgets — you don't need to worry about it,
# but it must be in the function signature.
def add_artist(b):
    with output:
        # clear_output() erases any previous message in the output area.
        # This stops old confirmations from piling up below the button.
        output.clear_output()

        # Read .value from each widget to get what the user typed or selected.
        artist_name = name_input.value
        artist_country = country_input.value
        artist_genre = genre_input.value

        # Open a fresh connection and cursor.
        # (In the exam, conn is usually opened in the setup cell above — you can reuse it.)
        conn2 = sqlite3.connect('moonrise_records.db')
        cursor2 = conn2.cursor()

        # The ? placeholders are filled in by the tuple (artist_name, artist_country, artist_genre).
        # This is safer than building the string by hand with f-strings,
        # because SQLite handles escaping special characters for you.
        cursor2.execute(
            "INSERT INTO artists (name, country, genre) VALUES (?, ?, ?)",
            (artist_name, artist_country, artist_genre)
        )

        # Don't forget to commit — without this, the new row is not saved.
        conn2.commit()
        conn2.close()

        # Print a confirmation message inside the output widget.
        print(f"Artist added: {artist_name} ({artist_country}, {artist_genre})")

# ─── Step 4: Connect the function to the button ─────────────────────────────
# on_click() tells the button which function to run when it is clicked.
submit_button.on_click(add_artist)

# ─── Step 5: Display the form ───────────────────────────────────────────────
# VBox() arranges widgets vertically (stacked on top of each other).
# HBox() arranges them horizontally (side by side).
# Here we stack all the inputs, then the button, then the output area.
form = VBox([
    name_input,
    country_input,
    genre_input,
    submit_button,
    output
])

display(form)

### Verifying the Insert

After using the form, run the cell below to confirm your new artist appears in the table. This is good practice in the exam — showing that your form actually worked is part of demonstrating the task.

In [ ]:
# Reconnect (in case the form used a separate connection above)
conn = sqlite3.connect('moonrise_records.db')

# .tail(3) shows the last 3 rows — the most recently added records
# appear at the bottom of the table.
df_artists = pd.read_sql("SELECT * FROM artists", conn)
display(df_artists.tail(3))

### Practice — Build a form for albums

Using the form above as your template, build a data entry form that inserts a new record into the **albums** table. The albums table has these columns: `title` (TEXT), `artist_id` (INTEGER), `year` (INTEGER), `price` (REAL), `stock` (INTEGER).

In [ ]:
# Your form here.
# Hint: use widgets.Text() for title,
#       widgets.IntText() for artist_id, year, and stock,
#       widgets.FloatText() for price.

# Remember the five steps:
# 1. Create the input widgets
# 2. Create the button and output area
# 3. Define the on_click function (with INSERT INTO)
# 4. Connect the function to the button with .on_click()
# 5. Display with VBox and display()


---

## Part 4: Creating a Chart with matplotlib (Task 4 in the Exam)

Task 4 always asks you to create a bar chart from data in the database. The process is:

1. Run a SQL query to get the data you need
2. Load the result into a pandas DataFrame with `pd.read_sql()`
3. Extract the two columns you want — one for the x-axis (labels) and one for the y-axis (values)
4. Use `plt.bar()` to draw the chart
5. Add a title and axis labels, then call `plt.tight_layout()` and `plt.show()`

### 4a: Bar Chart of Stock per Album

In [ ]:
# Step 1: Get the data from the database
df_stock = pd.read_sql("""
    SELECT title, stock
    FROM albums
    ORDER BY stock DESC
""", conn)

# Step 2: Draw the chart
# plt.figure(figsize=(10, 5)) sets the width and height of the chart in inches.
# The default size is often too small for charts with many bars.
plt.figure(figsize=(10, 5))

# plt.bar() takes the x-axis values first, then the y-axis values.
# df_stock['title'] gives us the album names column.
# df_stock['stock'] gives us the stock numbers column.
plt.bar(df_stock['title'], df_stock['stock'])

# Step 3: Add labels and a title
plt.title('Copies in Stock per Album')
plt.xlabel('Album')
plt.ylabel('Copies in Stock')

# Rotate x-axis labels so long album names don't overlap each other.
# ha='right' aligns the rotated text neatly to the right.
plt.xticks(rotation=45, ha='right')

# tight_layout() adjusts the spacing so nothing gets cut off at the edges.
# Always include this before plt.show().
plt.tight_layout()
plt.show()

### Practice — Part 4

Create a bar chart showing the number of albums in stock **per genre**. You will need to join the albums and artists tables to get the genre alongside the stock numbers, then use `COUNT(*)` or `SUM(stock)` to aggregate the data by genre.

If joins feel advanced, start with a simpler version: just show the price of each album as a bar chart.

In [ ]:
# Your chart here.
# Simpler version: create a bar chart of album prices (title vs price).
# Stretch version: join artists and albums, group by genre, sum stock.


---

## Part 5: Importing External Data (Task 5 in the Exam)

The final task asks you to read data from a CSV file and append it to a table in the database. The libraries involved are:

- `pd.read_csv()` — reads a CSV file into a pandas DataFrame
- `df.to_sql()` — writes a DataFrame into a SQLite table

### What is a CSV file?

A CSV (Comma-Separated Values) file is a plain text file where each line is a row of data and the values in each row are separated by commas. It is the simplest way to store a table as a file. In the exam, a CSV file of new records is provided and you are asked to import it into the database.

### The to_sql() function explained

`df.to_sql(name, con, if_exists, index)` writes a DataFrame to a database table.

The key parameters:
- `name` — the name of the table to write to (as a string)
- `con` — the database connection
- `if_exists='append'` — **append** adds new rows to the existing table. The other options are `'replace'` (wipe the table and start over) and `'fail'` (raise an error if the table already exists). In the exam you almost always want `'append'`.
- `index=False` — prevents pandas from writing its own row numbers into the table as an extra column

In [ ]:
# First, create a sample CSV file to import.
# In the exam this file is provided — you just need to read it.

csv_content = """name,country,genre
Buena Vista Social Club,Cuba,Son Cubano
Nick Drake,UK,Folk
Miriam Makeba,South Africa,Afropop
"""

with open('new_artists.csv', 'w') as f:
    f.write(csv_content)

print("new_artists.csv created.")

In [ ]:
# Step 1: Read the CSV file into a DataFrame
# pd.read_csv() takes the filename and returns a DataFrame.
# The first row of the CSV becomes the column names automatically.
df_new_artists = pd.read_csv('new_artists.csv')

print("Data from CSV:")
display(df_new_artists)

In [ ]:
# Step 2: Append the DataFrame to the artists table in the database
# Notice the columns in the CSV (name, country, genre) match the columns
# in the artists table — that's what makes this work.
# The artist_id column is missing from the CSV, which is fine:
# SQLite will generate IDs automatically because it is a PRIMARY KEY.

df_new_artists.to_sql(
    name='artists',        # name of the table to write to
    con=conn,              # the database connection
    if_exists='append',    # add to existing data, don't replace it
    index=False            # don't write pandas row numbers to the database
)

print("Import complete.")

In [ ]:
# Step 3: Verify the import worked
# Check the total number of artists now, and show the last few rows.

count_after = pd.read_sql("SELECT COUNT(*) AS total_artists FROM artists", conn)
print("Total artists after import:")
display(count_after)

print("Most recently added:")
df_check = pd.read_sql("SELECT * FROM artists", conn)
display(df_check.tail(5))

---

## Summary: The Complete Pattern

Every task in the exam follows one of these patterns. Knowing them by heart means you can focus on the specific details of the question rather than remembering syntax.

**Connecting to a database:**
```python
conn = sqlite3.connect('filename.db')
```

**Running a query and getting a DataFrame:**
```python
df = pd.read_sql("SELECT ... FROM ... WHERE ...", conn)
display(df)
```

**Checking table structure:**
```python
pd.read_sql("PRAGMA table_info(table_name)", conn)
```

**Inserting a single row directly:**
```python
cursor = conn.cursor()
cursor.execute("INSERT INTO table (col1, col2) VALUES (?, ?)", (val1, val2))
conn.commit()
```

**The ipywidgets form pattern (five steps):**
```python
# 1. Create widgets
field1 = widgets.Text(description='Label:')
btn = widgets.Button(description='Submit')
out = widgets.Output()

# 2. Define the on_click function
def on_submit(b):
    with out:
        out.clear_output()
        value = field1.value
        conn.cursor().execute("INSERT INTO ...", (value,))
        conn.commit()
        print("Done.")

# 3. Connect and display
btn.on_click(on_submit)
display(VBox([field1, btn, out]))
```

**Drawing a bar chart:**
```python
plt.figure(figsize=(10, 5))
plt.bar(df['label_column'], df['value_column'])
plt.title('Chart Title')
plt.xlabel('X Label')
plt.ylabel('Y Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
```

**Importing a CSV into the database:**
```python
df_new = pd.read_csv('filename.csv')
df_new.to_sql('table_name', conn, if_exists='append', index=False)
```

---

## Bibliography

Python Software Foundation. (2024). *sqlite3 — DB-API 2.0 interface for SQLite databases*. https://docs.python.org/3/library/sqlite3.html

pandas development team. (2024). *pandas documentation: Input/output*. https://pandas.pydata.org/docs/user_guide/io.html

Project Jupyter. (2024). *ipywidgets documentation*. https://ipywidgets.readthedocs.io/en/stable/

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://matplotlib.org

SQLite consortium. (2024). *SQLite documentation*. https://www.sqlite.org/docs.html